# Text Generation with LoRA via OpenVINO GenAI

LoRA, or [Low-Rank Adaptation](https://arxiv.org/abs/2106.09685), is a popular and lightweight training technique used for fine-tuning Large Language and Stable Diffusion Models without needing full model training. Full fine-tuning of larger models (consisting of billions of parameters) is inherently expensive and time-consuming. LoRA works by adding a smaller number of new weights to the model for training, rather than retraining the entire parameter space of the model. This makes training with LoRA much faster, memory-efficient, and produces smaller model weights (a few hundred MBs), which are easier to store and share.

At its core, LoRA leverages the concept of low-rank matrix factorization. Instead of updating all the parameters in a neural network, LoRA decomposes the parameter space into two low-rank matrices. This decomposition allows the model to capture essential information with fewer parameters, significantly reducing the amount of data and computation required for fine-tuning. This vastly reduces the storage requirement for large language models adapted to specific tasks and enables efficient task-switching during deployment all without introducing inference latency. 

![](https://github.com/user-attachments/assets/bf823c71-13b4-402c-a7b4-d6fc30a60d88)

Some more advantages of using LoRA:

* LoRA makes fine-tuning more efficient by drastically reducing the number of trainable parameters.
* The original pre-trained weights are kept frozen, which means you can have multiple lightweight and portable LoRA models for various downstream tasks built on top of them.
* LoRA is orthogonal to many other parameter-efficient methods and can be combined with many of them.
* Performance of models fine-tuned using LoRA is comparable to the performance of fully fine-tuned models.
* LoRA does not add any inference latency because adapter weights can be merged with the base model.

More details about LoRA can be found in HuggingFace [conceptual guide](https://huggingface.co/docs/peft/conceptual_guides/lora) and [blog post](https://huggingface.co/blog/peft).
  
In this tutorial we explore possibilities to use LoRA with OpenVINO Generative API.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Prepare models](#Prepare-models)
- [Select inference device](#Select-inference-device)
- [Create pipeline and generate results via OpenVINO GenAI without LoRA](#Create-pipeline-and-generate-results-via-OpenVINO-GenAI-without-LoRA)
- [Create pipeline and generate results via OpenVINO GenAI with LoRA](#Create-pipeline-and-generate-results-via-OpenVINO-GenAI-with-LoRA)
    - [Load adapter](#Load-adapter)
    - [Initialize pipeline with adapters and run inference](#Initialize-pipeline-with-adapters-and-run-inference)
    - [Get information about adapters](#Get-information-about-adapters)
    - [Disable adapters](#Disable-adapters)
    - [Remove adapter](#Remove-adapter)
    - [Selection specific adapter during generation](#Selection-specific-adapter-during-generation)
    - [Use several adapters](#Use-several-adapters)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-lora/llm-lora.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)


First, we should install the [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) for running model inference.

![](https://media.githubusercontent.com/media/openvinotoolkit/openvino.genai/refs/heads/master/src/docs/openvino_genai.svg)

[OpenVINO™ GenAI](https://github.com/openvinotoolkit/openvino.genai) is a library of the most popular Generative AI model pipelines, optimized execution methods, and samples that run on top of highly performant [OpenVINO Runtime](https://github.com/openvinotoolkit/openvino).

This library is friendly to PC and laptop execution, and optimized for resource consumption. It requires no external dependencies to run generative models as it already includes all the core functionality (e.g. tokenization via openvino-tokenizers).

In [ ]:
import platform

%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu "torch==2.8" "torchvision==0.23.0" "transformers==4.53.3" accelerate pillow "peft>=0.15.0"
%pip install -q "git+https://github.com/huggingface/optimum-intel.git"
%pip install -q -U "openvino>=2024.5.0" "openvino-tokenizers>=2024.5.0" "openvino-genai>=2024.5.0"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [15]:
import requests
from pathlib import Path

notebook_utils_path = Path("notebook_utils.py")

if not notebook_utils_path.exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    notebook_utils_path.open("w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-lora.ipynb")

## Prepare models
[back to top ⬆️](#Table-of-contents:)

As example, we will use already converted LLMs from [OpenVINO collection](https://huggingface.co/collections/OpenVINO/llm-6687aaa2abca3bbcec71a9bd). As example we will use [TinyLlama-1.1B-Chat-v1.0-int8-ov](https://huggingface.co/OpenVINO/TinyLlama-1.1B-Chat-v1.0-int8-ov).

In case, if you want run own models, you should convert them using [Hugging Face Optimum](https://huggingface.co/docs/optimum/intel/openvino/export) library accelerated by OpenVINO integration. More details about model preparation can be found in [OpenVINO LLM inference guide](https://docs.openvino.ai/2024/learn-openvino/llm_inference_guide/llm-inference-native-ov.html#convert-hugging-face-tokenizer-and-model-to-openvino-ir-format)

In [16]:
from pathlib import Path
import huggingface_hub as hf_hub

model_id = "OpenVINO/TinyLlama-1.1B-Chat-v1.0-int8-ov"

model_path = Path(model_id.split("/")[-1])

if not model_path.exists():
    hf_hub.snapshot_download(model_id, local_dir=model_path)

## Select inference device
[back to top ⬆️](#Table-of-contents:)


Select the device from dropdown list for running inference using OpenVINO.
> **Note**: For achieving maximal performance, we recommend to use GPU as target device if it is available.

In [17]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU", "AUTO"])

device

Dropdown(description='Device:', options=('CPU',), value='CPU')

## Create pipeline and generate results via OpenVINO GenAI without LoRA
[back to top ⬆️](#Table-of-contents:)

OpenVINO GenAI provides easy-to-use API for running text generation. Firstly we will create pipeline with `LLMPipeline`. `LLMPipeline` is the main object used for decoding. You can construct it straight away from the folder with the converted model. It will automatically load the `main model`, `tokenizer`, `detokenizer` and default `generation configuration`. 
After that we will configure parameters for decoding. 
Then we just run `generate` method and get the output in text format. We do not need to encode input prompt according to model expected template or write post-processing code for logits decoder, it will be done easily with LLMPipeline. 

In [18]:
import openvino_genai as ov_genai

pipe = ov_genai.LLMPipeline(model_path, device.value)

print(pipe.generate("Give me a sky blue color.", max_new_tokens=100))

Sure, here's a sky blue color:

Sky blue is a shade of blue that is typically associated with the sky or the ocean. It is a deep, rich blue color that is often used in fashion, interior design, and photography. Sky blue is a cool color, meaning it is cooler than other warm colors like red, orange, and yellow. It is also a classic color that is often associated with elegance, sophistication, and trust


## Create pipeline and generate results via OpenVINO GenAI with LoRA
[back to top ⬆️](#Table-of-contents:)

You can add one or multiple adapters into config and also specify alpha blending coefficients for their addition. OpenVINO GenAI supports LoRA adapters saved in Safetensors format. You can use one of publicly available pretrained adapters from [HuggingFace Hub](https://huggingface.co/models) or train your own.
> **Important Note**: Before loading pretrained adapters, please make sure that they are compatible with your base model architecture.

Generally, process of adapters configuration consists of 3 steps:
1. Load adapters, initialize pipeline with adapters and configure it. Use `openvino_genai.Adapter` to load LoRA. Use `openvino_genai.AdapterConfig` to initialize pipeline with adapters, add and remove adapters or change their weight coefficient for blending into pipeline.
2. Register adapters in pipeline constructor. These adapters will influence next generation. But you can also update `adapter_config`, remove some adapters or load a new one and pass it to `generate()` via adapters parameter.
3. Choose which adapter (or a combination of adapters) to apply in each `generate` call. It is not obligated to use all of provided in constructor adapters simultaneously, you can select one or combination of several among them for each generation cycle.

Adapter could be loaded in next mode:
* MODE_AUTO - Automatically selected
* MODE_DYNAMIC - A, B, alpha are fully variable
* MODE_STATIC_RANK - A and B have static shape, alpha is variable
* MODE_STATIC - A, B and alpha are constants
* MODE_FUSE - A, B and alpha are constants, fused to main matrix W

We will use default MODE_AUTO.

Loaded adapters could be added to `adapter_config` via `add(adapter, [alpha])` method or remove via `remove(adapter)`. You can change alpha by `set_alpha(aplha)`.

For more information, please, see LoRA adapters [user guide](https://github.com/openvinotoolkit/openvino.genai/blob/master/site/docs/guides/lora-adapters.mdx).

#### Load adapter
[back to top ⬆️](#Table-of-contents:)


Let's try [Javascript/tinyllama-colorist-lora](Javascript/tinyllama-colorist-lora), which was trained on color dataset to fine-tune TinyLLama to be a colorist expert.

In [19]:
from huggingface_hub import hf_hub_download

lora_dir = Path("lora")

colorista_lora_id = "Javascript/tinyllama-colorist-lora"
colorita_lora_path = lora_dir / "tinyllama-colorist-lora"

if not colorita_lora_path.exists():
    hf_hub_download(repo_id=colorista_lora_id, filename="adapter_model.safetensors", local_dir=colorita_lora_path)

#### Initialize pipeline with adapters and run inference
[back to top ⬆️](#Table-of-contents:)

In [20]:
adapter_config = ov_genai.AdapterConfig()

colorist_adapter = ov_genai.Adapter(colorita_lora_path / "adapter_model.safetensors")
adapter_config.add(colorist_adapter, alpha=0.5)

pipe_with_adapters = ov_genai.LLMPipeline(model_path, device.value, adapters=adapter_config)
print(pipe_with_adapters.generate("Give me a sky blue color.", max_new_tokens=100))

Sure, here's a sky blue color:

- RGB: 145, 206, 230
- Hex: #c3e6ff
- HSL: 120°, 100%, 50%
- HSV: 120°, 100%, 50%
- CMYK: 0%, 0%, 100%, 


### Get information about adapters
[back to top ⬆️](#Table-of-contents:)

You can get all loaded adapters via `get_adapters()`. To find out what alpha value is used for a particular adapter, it could be used `get_alpha(adapter)`.

In [21]:
print("Loaded adapters numers: ", len(adapter_config.get_adapters()))
print("Alpha for colorist adapter: ", adapter_config.get_alpha(colorist_adapter))

adapter_config.get_adapters()

Loaded adapters numers:  1
Alpha for colorist adapter:  0.5


### Disable adapters
[back to top ⬆️](#Table-of-contents:)

You can disable adapters providing empty `AdapterConfig` into generate

In [22]:
print(pipe_with_adapters.generate("Give me a sky blue color.", max_new_tokens=100, adapters=ov_genai.AdapterConfig()))

Sure, here's a sky blue color:

Sky blue is a shade of blue that is typically associated with the sky or the ocean. It is a deep, rich blue color that is often used in fashion, interior design, and photography. Sky blue is a cool color, which means it is less warm and more cool than warm colors like red or orange. It is often used in the summer months, when the sky is often blue and the weather is warm.


### Remove adapter
[back to top ⬆️](#Table-of-contents:)


In [23]:
adapter_config.remove(colorist_adapter)
print("Loaded adapters: ", len(adapter_config.get_adapters()))

Loaded adapters:  0


Let's try more adapters, which fine-tune TinyLLama: [snshrivas10/sft-tiny-chatbot](https://huggingface.co/snshrivas10/sft-tiny-chatbot) and [emilykang/medprob-anatomy_lora](https://huggingface.co/emilykang/medprob-anatomy_lora)

In [25]:
chatbot_lora_id = "snshrivas10/sft-tiny-chatbot"
chatbot_lora_path = lora_dir / "sft-tiny-chatbot"

if not chatbot_lora_path.exists():
    hf_hub_download(repo_id=chatbot_lora_id, filename="adapter_model.safetensors", local_dir=chatbot_lora_path)

med_lora_id = "therealcyberlord/TinyLlama-1.1B-Medical"
med_lora_path = lora_dir / "TinyLlama-1.1B-Medical"

if not med_lora_path.exists():
    hf_hub_download(repo_id=med_lora_id, filename="adapter_model.safetensors", local_dir=med_lora_path)

In [26]:
chatbot_adapter = ov_genai.Adapter(chatbot_lora_path / "adapter_model.safetensors")
adapter_config.add(chatbot_adapter)

print("Loaded adapters: ", len(adapter_config.get_adapters()))
print("Alpha for chatbot adapter: ", adapter_config.get_alpha(chatbot_adapter))

Loaded adapters:  1
Alpha for chatbot adapter:  1.0


### Selection specific adapter during generation
[back to top ⬆️](#Table-of-contents:)

Providing adapters argument with `openvino_genai.AdapterConfig` into `generate` allow to select one or several from them.

In [27]:
print(pipe_with_adapters.generate("Give me a sky blue color.", max_new_tokens=100, adapters=adapter_config))

A sky blue color is a shade of blue that is a deep, rich blue with a hint of green. It is a beautiful and calming color that is often associated with nature and peacefulness.


### Use several adapters
[back to top ⬆️](#Table-of-contents:)


Let's add one more adapter to `adapter_config` and put it into `generate()`.

In [28]:
med_adapter = ov_genai.Adapter(med_lora_path / "adapter_model.safetensors")
adapter_config.add(med_adapter)

print("Loaded adapters: ", len(adapter_config.get_adapters()))
print("Alpha for medprob-anatomy_lora adapter: ", adapter_config.get_alpha(med_adapter))

Loaded adapters:  2
Alpha for medprob-anatomy_lora adapter:  1.0


In [29]:
print(pipe_with_adapters.generate("What is the structure of the frontal lobe of the brain?", max_new_tokens=100, adapters=adapter_config))

The frontal lobe of the brain is a part of the brain that is responsible for decision-making, attention, and impulse control. It is located on the front of the brain and is divided into two halves: the prefrontal cortex and the anterior cingulate cortex. The prefrontal cortex is responsible for higher-level cognitive functions such as planning, reasoning, and decision-making, while the anterior cingulate cortex is responsible for regulating
